In [5]:
%%writefile hola.cpp

#include <iostream>
using namespace std;

void swapArith(int &u, int &v) {
    u = u + v;
    v = u - v;
    u = u - v;
}


int main() {
    int a = 15, b = 42;
    cout << "antes u=" << a << " v=" << b;
    swapArith(a, b);
    cout << " | despues u=" << a << " v=" << b << endl;
    return 0;
}

Overwriting hola.cpp


In [6]:
!g++ hola.cpp -o hola
!./hola

antes u=15 v=42 | despues u=42 v=15


In [9]:
%%writefile hola.cpp
#include <iostream>
#include <vector>
using namespace std;

vector<int> comparisonCountingSort(const vector<int> &A) {
    int n = A.size();
    vector<int> count(n, 0), S(n);
    for (int i = 0; i < n - 1; i++)
        for (int j = i + 1; j < n; j++)
            if (A[i] < A[j]) count[j]++;
            else             count[i]++;
    for (int i = 0; i < n; i++) S[count[i]] = A[i];
    return S;
}

void print(const char *label, const vector<int> &v) {
    cout << label;
    for (int x : v) cout << x << " ";
    cout << endl;
}

int main() {
    vector<int> tests[] = {
        {62, 31, 84, 96, 19, 47},
        {5, 2, 9, 2, 5, 1, 9},
        {9, 8, 7, 6, 5, 4, 3, 2, 1, 0}
    };
    for (auto &t : tests) {
        print("Entrada : ", t);
        print("Salida  : ", comparisonCountingSort(t));
        cout << endl;
    }
    return 0;
}

Overwriting hola.cpp


In [10]:
!g++ hola.cpp -o hola
!./hola

Entrada : 62 31 84 96 19 47 
Salida  : 19 31 47 62 84 96 

Entrada : 5 2 9 2 5 1 9 
Salida  : 1 2 2 5 5 9 9 

Entrada : 9 8 7 6 5 4 3 2 1 0 
Salida  : 0 1 2 3 4 5 6 7 8 9 



In [12]:
%%writefile hola.cpp

#include <iostream>
#include <vector>
#include <chrono>
#include <cstdio>
#include <cstdlib>
using namespace std;

vector<int> comparisonCountingSort(const vector<int> &A) {
    int n = A.size();
    vector<int> count(n, 0), S(n);
    for (int i = 0; i < n - 1; i++)
        for (int j = i + 1; j < n; j++)
            if (A[i] < A[j]) count[j]++; else count[i]++;
    for (int i = 0; i < n; i++) S[count[i]] = A[i];
    return S;
}

vector<int> distributionCountingSort(const vector<int> &A, int l, int u) {
    int n = A.size(), k = u - l + 1;
    vector<int> D(k, 0), S(n);
    for (int i = 0; i < n; i++) D[A[i] - l]++;
    for (int j = 1; j < k; j++) D[j] += D[j - 1];
    for (int i = n - 1; i >= 0; i--) {
        int j = A[i] - l;
        S[D[j] - 1] = A[i];
        D[j]--;
    }
    return S;
}

void print(const char *label, const vector<int> &v) {
    cout << label;
    for (int x : v) cout << x << " ";
    cout << endl;
}

int main() {
    vector<int> A = {13, 11, 12, 13, 12, 12};         // rango 11..13
    print("Distribution: ", distributionCountingSort(A, 11, 13));
    print("Comparison  : ", comparisonCountingSort(A));

    cout << "\n      n | Comparison (ms) | Distribution (ms) | iguales\n";
    srand(42);
    for (int n : {1000, 5000, 20000}) {
        vector<int> B(n);
        for (auto &x : B) x = rand() % 100;
        auto t0 = chrono::steady_clock::now();
        vector<int> s1 = comparisonCountingSort(B);
        auto t1 = chrono::steady_clock::now();
        vector<int> s2 = distributionCountingSort(B, 0, 99);
        auto t2 = chrono::steady_clock::now();
        double m1 = chrono::duration<double, milli>(t1 - t0).count();
        double m2 = chrono::duration<double, milli>(t2 - t1).count();
        printf("%7d | %15.3f | %17.3f | %s\n", n, m1, m2, s1 == s2 ? "si" : "NO");
    }
    return 0;
}

Overwriting hola.cpp


In [13]:
!g++ hola.cpp -o hola
!./hola

Distribution: 11 12 12 12 13 13 
Comparison  : 11 12 12 12 13 13 

      n | Comparison (ms) | Distribution (ms) | iguales
   1000 |           9.735 |             0.040 | si
   5000 |         209.680 |             0.164 | si
  20000 |        4748.059 |             0.961 | si


In [16]:
%%writefile hola.cpp

#include <iostream>
using namespace std;

class HashTable {
    enum State { EMPTY, USED, DELETED };
    int *keys;
    State *state;
    int m, count;
    int h(int k) const { return ((k % m) + m) % m; }

public:
    HashTable(int size) : m(size), count(0) {
        keys = new int[m];
        state = new State[m];
        for (int i = 0; i < m; i++) state[i] = EMPTY;
    }
    ~HashTable() { delete[] keys; delete[] state; }

    int search(int k) const {
        int i = h(k);
        for (int p = 0; p < m; p++, i = (i + 1) % m) {
            if (state[i] == EMPTY) return -1;
            if (state[i] == USED && keys[i] == k) return i;
        }
        return -1;
    }

    bool insert(int k) {
        if (search(k) >= 0) return true;
        if (count == m) return false;
        int i = h(k);
        while (state[i] == USED) i = (i + 1) % m;
        keys[i] = k; state[i] = USED; count++;
        return true;
    }

    bool remove(int k) {
        int i = search(k);
        if (i < 0) return false;
        state[i] = DELETED; count--;
        return true;
    }

    void print() const {
        for (int i = 0; i < m; i++) {
            cout << "  [" << i << "] ";
            if (state[i] == USED) cout << keys[i];
            else if (state[i] == DELETED) cout << "(borrado)";
            else cout << "-";
            cout << endl;
        }
    }
};

int main() {
    HashTable t(11);
    int data[] = {10, 22, 31, 4, 15, 28, 17, 88, 59};
    for (int i = 0; i < 9; i++) t.insert(data[i]);
    cout << "Tabla (m = 11, h(k) = k mod 11):" << endl;
    t.print();
    return 0;
}

Overwriting hola.cpp


In [17]:
!g++ hola.cpp -o hola
!./hola

Tabla (m = 11, h(k) = k mod 11):
  [0] 22
  [1] 88
  [2] -
  [3] -
  [4] 4
  [5] 15
  [6] 28
  [7] 17
  [8] 59
  [9] 31
  [10] 10
